## This analysis is concerned with the enrichment of LASV/Mammarenavirus/Arenavirus-specific peptides by LASV monoclonal antibodies. 

In [198]:
import pandas as pd
df = pd.read_csv('downloads_study920_mapped_rpk.csv').set_index('peptide')
peps_metadata = pd.read_csv('../metadata_files/lassa_library_sequences_species_standardized_070726_ICTV_corrected.csv')

In [199]:
df_mAb = df.filter(regex='mAb')
df_AG = df.filter(regex='AG')

In [200]:
df_AG_avg = df_AG.mean(axis=1)

In [201]:
df_mAb_fc = (df_mAb + 1).div((df_AG_avg + 1), axis=0)

### filter to Arenavirus peptides and glycoprotein

In [202]:
peps_metadata_arenavirus_gp = peps_metadata.loc[(peps_metadata['family'] == 'Arenaviridae') & (peps_metadata['protein_std'] == 'glycoprotein')]
seq_id = peps_metadata_arenavirus_gp['seq_id']

In [203]:
df_mAb_fc_arenaviridae_gp = df_mAb_fc.loc[seq_id]

## now going to investigate each mAb individually - starting with mAbs known to bind linear epitopes (and thus being non-neutralizing)

In [204]:
import numpy as np
import pandas as pd
from Bio import Align, SeqIO
from Bio.Align import substitution_matrices 

def analyze_mab_peptides(
    df, 
    peps_metadata, 
    regex, 
    ref_path="../ref_seqs/UP000002473_11622_LASV_Josiah_reference.fasta", 
    target_id="sp|P08669|GLYC_LASSJ", 
    top_n=20, 
    literature_note=None, 
    print_alignments=True
):
    """Select two mAb replicate columns using regex, calculate Pearson's r, only keep if passes 0.95,
    rank peptides by mean enrichment, and align the top peptides to a reference protein. 
    
    Reference coordinates are 1-based and inclusive. 
    Discontinuous covered regions are reported separately. 
    """
    def get_covered_regions(positions):
        """Convert positions into separate contiguous intervals"""
        positions = sorted(set(positions))

        if not positions:
            return[]

        regions = []
        region_start = positions[0]
        region_end = positions[0]

        for position in positions[1:]:

            if position == region_end + 1:
                region_end = position #extend the current region
            else:
                #gap: save the completed region
                regions.append((region_start, region_end))

                #start a new region at the current position
                region_start = position
                region_end = position

        regions.append((region_start, region_end))
        return regions


    """calculate pearson's R for Arenavirus GPC peptides and find top 20 Arenavirus GPC peptides"""
    mAbs = df.filter(regex=regex, axis=1).copy()
    
    mAbs = (mAbs.apply(pd.to_numeric, errors='raise'))

    pearsons_r = mAbs.iloc[:, 0].corr(mAbs.iloc[:, 1], method='pearson')

    if pearsons_r < 0.95: 
        print(f"Correlation too low, Analysis stopped")

        #return exits the function
        return { 
            "pearsons_r": pearsons_r, 
            "passed_correlation": False, 
            "top_peptides": None, 
            "mapping": None,
            "covered_regions": [], 
            "positions_covered": []
        }
        
    ### ---- Rank peptides by mean enrichment 

    mAbs_avg = mAbs.mean(axis=1)

    mAbs_top = mAbs_avg.dropna().sort_values(ascending=False).head(top_n).rename("mean_fc")


    ### --- retrieve metadata 
    mAbs_top_pep_sequences = peps_metadata.loc[peps_metadata['seq_id'].isin(mAbs_top.index.to_list())]

    """alignment part""" #---- Load reference

    #SeqIO.to_dict() expects parsed sequence records
    ref_records = SeqIO.to_dict(SeqIO.parse(ref_path, 'fasta'))

    target_id = "sp|P08669|GLYC_LASSJ"
    ref_seq = ref_records[target_id].seq

    #------- configure pairwise aliger for peptides 

    aligner = Align.PairwiseAligner()
    aligner.mode = 'local' #for peptides
    aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
    aligner.open_gap_score = -10
    aligner.extend_gap_score = -0.5

    #align each peptide to the reference sequence
    peptides = mAbs_top_pep_sequences['sequence'].to_list()

    positions_covered = set()

    for _, row in mAbs_top_pep_sequences.iterrows():

        peptide_id = row['seq_id']

        pep = str(row['sequence']).strip().upper()

        alignments = aligner.align(ref_seq, pep)
        best_alignment = alignments[0] #top scoring alignment

        if best_alignment is None: 
            continue

        #All aligned blocks; convert to 1-based inclusive positions
        for ref_start, ref_end in best_alignment.aligned[0]:
            positions_covered.update(
                range(int(ref_start) + 1, int(ref_end) +1) #inside range() need ref_end+1, because python excludes the stopping value
            )
        
        if print_alignments:
            print(f"\nPeptide ID: {peptide_id}")
            print(f"Peptide: {pep} | Score: {best_alignment.score}")
            print(best_alignment)


    covered_regions = get_covered_regions(positions_covered)

    print(f"\nResults for {regex}")
    print(f"Pearson's r = {pearsons_r:.3f}")
    print(f"Top peptides selected {len(mAbs_top)}")
    print(f"Covered reference regions (1-based, inclusive) {covered_regions}")


    if literature_note:
        print(literature_note)

    return {
        "pearsons_r": pearsons_r,
        "top_peptides": peptides,
        "peptides enrichment values": mAbs_top,
        "covered_regions": covered_regions,
        "positions_covered": sorted(positions_covered),
    }


    

In [205]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='1-1B', literature_note = "Literature reports LASV 1.1B to bind a linear epitope within the T-loop (GP2-L2) of the GP2 subunit of GPC, mapped to residues aa361-375, non-neutralizing")


Peptide ID: XOB76283.1|precursor|Yunnan_rodent_mammarenavirus_1|HHJP-YNSC97|Niviventer_fulvescens|China:_Yunnan|2021|?|?|tile_17
Peptide: KFWYLNNTKTGQVSLPQCWLISNGSYLNETHFSADIEREANNLITE | Score: 204.0
target          367 KYWYLNHTTTGRTSLPKCWLVSNGSYLNETHFSDDIEQQADNMITE 413
                  0 |.||||.|.||..|||.|||.||||||||||||.|||..|.|.|||  46
query             0 KFWYLNNTKTGQVSLPQCWLISNGSYLNETHFSADIEREANNLITE  46


Peptide ID: XOB76280.1|glycoprotein|Mammarenavirus_ryukyuense|CXMIX-YNSF117|Mus_musculus|China:_Yunnan|2021|S|?|tile_16
Peptide: TVNSLISDQLLMRNHLRDLMGVPYCNYSKFWYLEHAKTGETSVPKC | Score: 208.0
target          340 VNALINDQLIMKNHLRDIMGIPYCNYSKYWYLNHTTTGRTSLPKC 385
                  0 ||.||.|||.|.|||||.||.|||||||.|||.|..||.||.|||  45
query             1 VNSLISDQLLMRNHLRDLMGVPYCNYSKFWYLEHAKTGETSVPKC  46


Peptide ID: XOB76280.1|glycoprotein|Mammarenavirus_ryukyuense|CXMIX-YNSF117|Mus_musculus|China:_Yunnan|2021|S|?|tile_17
Peptide: YCNYSKFWYLEHAKTGETSVPKCWLVTNGSYLNETHFSDQIEQEAD | Sco

{'pearsons_r': np.float64(0.9970396135453448),
 'top_peptides': ['KFWYLNNTKTGQVSLPQCWLISNGSYLNETHFSADIEREANNLITE',
  'TVNSLISDQLLMRNHLRDLMGVPYCNYSKFWYLEHAKTGETSVPKC',
  'YCNYSKFWYLEHAKTGETSVPKCWLVTNGSYLNETHFSDQIEQEAD',
  'TINSLISDQLLMRNHLRDLMGVPYCNYSKFWYLEHSKTNETSVPKC',
  'KFWYLNNTKTGQVSLPQCWLISNGSYLNKTDFTDDIEREANNLVTE',
  'SKFWYLNDTRNGKTSLPKCWLISNGSYLNETHFSDQIEQEADNMIT',
  'YSKFWYLEHYKTGETSLPKCWLVSNGSYLNETHFSTEIEQEADNMI',
  'YTKFWFLNDTRTGRTSLPKCWKVSNNTYLNETEFSTEIEQEADNMI',
  'LKELLDTPYCNYTKFWYVNHTITGEHSLPRCWMVKNNSYLNESEFR',
  'NYSKFWYLNDTRNGRTSLPKCWLVSNGSYLNETQFSDTIEQEADNM',
  'NDQLIMKNHLRDIMGIPYCNYSKYWYLNNTVTGKSSLPKCWLVSNG',
  'YSKFWYLEHYKTGETSLPKCWLVSNGSYLNETHFSDEIEQEANNMI',
  'KQLARIPYCNYTKFWYINDTVSGKHSLPQCWKVHNGSYLNESQFKN',
  'TRFWYLNNTKTGQVSLPKCWLISNGSYLNQTSFTDDIEREANNLIT',
  'TVTPLISDQLLMRNHLRDLMGVPYCNSSKFWYLEHAKTGETSVPKC',
  'TVNSLISDQLLMRNHLRDLMGVPYCNYSKFWYLEHVKTKETSVPKC',
  'YCNYSKFWYLKHVKTGETSVPKCWLISNGSYLNETHFSDQIEQEAD',
  'YCNYSKFWYLEHSRTGESSTPKCWLVSTGSYLNETHFSDQIEQEAD',
 

In [206]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='13-4E', literature_note = "Literature reports LASV 13-4E to bind a linear epitope within the T-loop (GP2-L2) of the GP2 subunit of GPC, mapped to residues aa361-375, non-neutralizing")


Peptide ID: XOB76283.1|precursor|Yunnan_rodent_mammarenavirus_1|HHJP-YNSC97|Niviventer_fulvescens|China:_Yunnan|2021|?|?|tile_17
Peptide: KFWYLNNTKTGQVSLPQCWLISNGSYLNETHFSADIEREANNLITE | Score: 204.0
target          367 KYWYLNHTTTGRTSLPKCWLVSNGSYLNETHFSDDIEQQADNMITE 413
                  0 |.||||.|.||..|||.|||.||||||||||||.|||..|.|.|||  46
query             0 KFWYLNNTKTGQVSLPQCWLISNGSYLNETHFSADIEREANNLITE  46


Peptide ID: XOB76280.1|glycoprotein|Mammarenavirus_ryukyuense|CXMIX-YNSF117|Mus_musculus|China:_Yunnan|2021|S|?|tile_17
Peptide: YCNYSKFWYLEHAKTGETSVPKCWLVTNGSYLNETHFSDQIEQEAD | Score: 225.0
target          362 YCNYSKYWYLNHTTTGRTSLPKCWLVSNGSYLNETHFSDDIEQQAD 408
                  0 ||||||.|||.|..||.||.||||||.||||||||||||.|||.||  46
query             0 YCNYSKFWYLEHAKTGETSVPKCWLVTNGSYLNETHFSDQIEQEAD  46


Peptide ID: XDC94394.1|precursor|Mammarenavirus_sp.|EriAreV/RU/MOS/2022/hed22_15/S-v1|Erinaceus_sp.|Russia|2022|S|?|tile_8
Peptide: KELLGIPYCNYTKFWYLNNTSLNLHSLPKCWLVSNNSYLNKTEFSF

{'pearsons_r': np.float64(0.9983442255779336),
 'top_peptides': ['KFWYLNNTKTGQVSLPQCWLISNGSYLNETHFSADIEREANNLITE',
  'YCNYSKFWYLEHAKTGETSVPKCWLVTNGSYLNETHFSDQIEQEAD',
  'KELLGIPYCNYTKFWYLNNTSLNLHSLPKCWLVSNNSYLNKTEFSF',
  'NDQLIMKNHLRDIMGIPYCNYSRYWYLNHTTTGKTSLPKCWLVSNG',
  'INDQLIMRNHLKDLMGIPYCNYTKFWYLNNTKTGQVSLPQCWLISN',
  'KFWYLNNTKTGQVSLPQCWLISNGSYLNKTDFTDDIEREANNLVTE',
  'RFWYLNNTRTGQVSLPQCWLITNGSYLNKTEFTNDIEREANNLVTE',
  'SKFWYLNDTRNGKTSLPKCWLISNGSYLNETHFSDQIEQEADNMIT',
  'DQLIMKNHLRDIMGIPYCNYSRYWYLNHTSTGKTSLPRCWLISNGS',
  'NYSKFWYLNDTRNGRTSLPKCWLVSNGSYLNETQFSDTIEQEADNM',
  'NDQLIIKNHLRDLMGIPYCNYSKFWYLNDTRNGKTSLPKCWKVSNN',
  'LGIPYCNYTKFWYLNSTRTGHHSLPKCWMISNGSYLNETHFSTEIE',
  'DQLIMKNHLRDIMGIPYCNYSKYWYLNHTSSGRTSLPKCWLVSNGS',
  'DQLIMKNHLRDIMGIPYCNYSKYWYLNHTITGKTSLPRCWLVSNGS',
  'DQLIMKNHLRDIMGIPYCNYSKYWYLNHTVTGKTSLPKCWLVSNGS',
  'NDQLIMKNHLRDIMGIPYCNYSKYWYLNNTVTGKSSLPKCWLVSNG',
  'YCNYTKFWYLNSTKTGVVSLPKCWLVSNGSYLNVTHFSNDIEQEAD',
  'TRFWYLNNTKTGQVSLPKCWLISNGSYLNQTSFTDDIEREANNLIT',
 

In [207]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='22-5D', literature_note = "Literature reports LASV 22-5D to bind linear-epitope region (GP2-L1) between fusion loop (aa279-295) and HR1 (aa295-363)")


Peptide ID: XOB76280.1|glycoprotein|Mammarenavirus_ryukyuense|CXMIX-YNSF117|Mus_musculus|China:_Yunnan|2021|S|?|tile_14
Peptide: GNTAVAKCNMNHDEEFCDMLRLIDYNKAALTKFKTDVESALHLFKT | Score: 151.0
target          293 GNTAVAKCNEKHDEEFCDMLRLFDFNKQAIQRLKAEAQMSIQL 336
                  0 |||||||||..|||||||||||.|.||.|....|........|  43
query             0 GNTAVAKCNMNHDEEFCDMLRLIDYNKAALTKFKTDVESALHL  43


Peptide ID: XDC94394.1|precursor|Mammarenavirus_sp.|EriAreV/RU/MOS/2022/hed22_15/S-v1|Erinaceus_sp.|Russia|2022|S|?|tile_5
Peptide: ASELKCFGNTAIAKCNTNHDEEFCDMLSLIEYNKKALIQFKDNLRE | Score: 160.0
target          287 AELKCFGNTAVAKCNEKHDEEFCDMLRLFDFNKQAIQRLK 327
                  0 .|||||||||.||||..|||||||||.|...||.|....|  40
query             1 SELKCFGNTAIAKCNTNHDEEFCDMLSLIEYNKKALIQFK  41


Peptide ID: BDX23356.1|precursor|Mammarenavirus_guanaritoense|?|Homo_sapiens|Venezuela|1997-09-02|S|S-26764|tile_13
Peptide: KCFGNTAVAKCNLNHDSEFCDMLRLFDFNKNAIEKLNNQTKTAVNM | Score: 175.0
target          290 KCFG

{'pearsons_r': np.float64(0.9945907992610531),
 'top_peptides': ['GNTAVAKCNMNHDEEFCDMLRLIDYNKAALTKFKTDVESALHLFKT',
  'ASELKCFGNTAIAKCNTNHDEEFCDMLSLIEYNKKALIQFKDNLRE',
  'KCFGNTAVAKCNLNHDSEFCDMLRLFDFNKNAIEKLNNQTKTAVNM',
  'LKCFGNTAVAKCNLNHDSEFCDMLRLFDYNKQAILRLKADLETTLE',
  'AIAKCNLNHDEEFCDMLRLFDFNKQAIDKLRIEAQRGLNLITKAVN',
  'TAVAKCNLNHDEEFCDMLRLIDFNKQAVRRLKNETQLGLRLVTKAV',
  'CNEKHDEEFCDMLRLFDFNKQAIKRLKTEAQMNIQLINKAVNALIN',
  'VAKCNLNHDEEFCDMLRLFDFNKQAIDKLRIEAQRGLNLITKAVNS',
  'WAGIKCFDNSVMAKCNKDHNEEFCDTMRLFDFNQNAIKTLQLNTEN',
  'LKCFGNTALAKCNLKHDSEFCDMIKLFDFNKNAISKLNNNTIEAVN',
  'ASELKCFGNTAVAKCNINHDSEFCDMLRLFDYNKKAIVNLQDKTKA',
  'TAIAKCNLNHDEEFCDMLRLIDFNKQALKTFKSEVNHGLQLITKAI',
  'IAKCNQNHDSEFCDMLRLFDYNKNAIKTLNEETKTRVNVLSHTINA',
  'ANCNEKHDSEFCDMLRLFDFNREAIKRLKIEANKSINLITSAVNSL',
  'AELKCFGNTAVAKCNEKHDEEFCDMLRLFDFNKQAISRLRSEAQMS',
  'SELKCFGNTAIAKCNLDHNSEFCDMLRLFEFNKNAIKMLQNESKHQ',
  'MLLAAEMKCFGNTAIAKCNQNHDSEFCDMLRLFDYNKNAIKTLNEE',
  'CLTRSMLIGMDLKCFGNTAIAKCNQKHDEEFCDMLRLFDFNKQAIS',
 

In [208]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='24-6C', literature_note = "Literature reports LASV 24-6C to bind linear-epitope region (GP2-L3) HR2 region, aa400-412")


Peptide ID: XBA96065.1|precursor|Mammarenavirus_lassaense|LF2350|Homo_sapiens|Sierra_Leone|2012|S|?|tile_17
Peptide: YWYLNHTTTGKTSLPKCWLVSNGSYLNETHFSDDIEQQADNMITEM | Score: 255.0
target          368 YWYLNHTTTGRTSLPKCWLVSNGSYLNETHFSDDIEQQADNMITEM 414
                  0 ||||||||||.|||||||||||||||||||||||||||||||||||  46
query             0 YWYLNHTTTGKTSLPKCWLVSNGSYLNETHFSDDIEQQADNMITEM  46


Peptide ID: XBA96065.1|precursor|Mammarenavirus_lassaense|LF2350|Homo_sapiens|Sierra_Leone|2012|S|?|tile_18
Peptide: SYLNETHFSDDIEQQADNMITEMLQKEYMERQGKTPLGLVDLFVFS | Score: 237.0
target          391 SYLNETHFSDDIEQQADNMITEMLQKEYMERQGKTPLGLVDLFVFS 437
                  0 ||||||||||||||||||||||||||||||||||||||||||||||  46
query             0 SYLNETHFSDDIEQQADNMITEMLQKEYMERQGKTPLGLVDLFVFS  46


Peptide ID: WAD86895.1|glycoprotein|Mammarenavirus_lassaense|?|Homo_sapiens|Nigeria|1977|S|GA391|tile_17
Peptide: WYLNHTSTGKTSLPRCWLISNGSYLNETKFSDDIEQQADNMITEML | Score: 235.0
target          369 WYLNHTTTGRTSLPK

{'pearsons_r': np.float64(0.9847683815443297),
 'top_peptides': ['YWYLNHTTTGKTSLPKCWLVSNGSYLNETHFSDDIEQQADNMITEM',
  'SYLNETHFSDDIEQQADNMITEMLQKEYMERQGKTPLGLVDLFVFS',
  'WYLNHTSTGKTSLPRCWLISNGSYLNETKFSDDIEQQADNMITEML',
  'YLNETKFSDDIEQQADNMITEMLQKEYIDRQGKTPLGLVDLFVFST',
  'WYLNHTTTGRTSLPRCWLVSNGSYLNETHFSDDIEQQADNMITEML',
  'WYLNHTSSGRTSLPKCWLVSNGSYLNETHFSDDIEQQADNMITEML',
  'WYLNHTVTGKTSLPRCWLVSNGTYLNETHFSDDIEQQADNMITELL',
  'YLNETQFSDDIEQQADNMITELLQKEYMDRQGKTPLGLVDLFVFST',
  'WYLNHTVTGRTSLPRCWLVSNGSYLNETHFSDDIEQQADNMITELL',
  'WYLNHTVTGKTSLPKCWLVSNGSYLNETHFSDDIEQQADNMITELL',
  'YLNETHFSDDIEQQADNMITELLQKEYIDRQGKTPLGLVDLFVFST',
  'SYLNETHFSEDIEQQADNMITEMLQKEYIDRQGKTPLGLVDIFIFS',
  'YLNETHFSDDIEQQADNMITEMLQKEYLDRQGKTPLGLVDLFVFST',
  'ISNGTYLNETHFSDEIEQEADNMITEMLRKEYQERQGKTPLGLVDL',
  'WYLNHTVTGKTSLPRCWLVSNGSYLNETYFSDDIEQQADNMITELL',
  'YLNETYFSDDIEQQADNMITELLQKEYIDRQGRTPLGLVDLFVFST',
  'WYLNNTTTGRTSLPRCWLISNGSYLNETHFSDDIEQQADNMITEML',
  'TSLPKCWLVSNGSYLNETHFSDDIEQQADNMITEMLQKEYMDRQGK',
 

### now looking at GPC-B mAbs - first neutralizing

In [209]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='18-5C', literature_note = "Literature reports LASV 18.5C to bind a quaternary pre-fusion epitope. Site A: GP1 N-terminal loop \
aa62-63, GP2 HR2 aa356-362, GP2 T-Loop and HR2 aa387-408. N395 (with glycan), H398, Q405, and D408 sit directly at the binding interface. \
site B: GP2 fusion peptide aa266-275, and GP2 HR1 aa324-326. Unlike other GPC-B antibodies, 18.5C does not form hydrogen bonds with T397 or H398. \
Note that this mAb had overall weaker neutralization capacities than 25.6A and 37.7H, in addition to poorer binding to the LASV gpc. Introduction of argininges in its CDRs helped"
)

Correlation too low, Analysis stopped


{'pearsons_r': np.float64(-0.0023287692299385457),
 'passed_correlation': False,
 'top_peptides': None,
 'mapping': None,
 'covered_regions': [],
 'positions_covered': []}

In [210]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='^25-6A_', literature_note = "Literature reports LASV 25.6A to bind a quaternary pre-fusion epitope. Site A: GP1 N-terminus \n \
GP2 T-loop, HR1 and HR2. GP1 subunit aa62-63, GP2 subunit aa356-352, 387-408. Site B: GP2 subunit aa266-275 and aa324-326. \n \
key interactions: T-loop hydrogen bond network: with N395, E396, T397, H398, D402. H398 central anchor with 3 bonds to the antibody CDRs \n \
HR2 contacts R36 and R108, heavy chain argininges form hydrogen bonds with GP2 Q405 and D408 \n \
antibody footprint interacts with glycans attached to N79, N390, N395")


Peptide ID: XOB76280.1|glycoprotein|Mammarenavirus_ryukyuense|CXMIX-YNSF117|Mus_musculus|China:_Yunnan|2021|S|?|tile_14
Peptide: GNTAVAKCNMNHDEEFCDMLRLIDYNKAALTKFKTDVESALHLFKT | Score: 151.0
target          293 GNTAVAKCNEKHDEEFCDMLRLFDFNKQAIQRLKAEAQMSIQL 336
                  0 |||||||||..|||||||||||.|.||.|....|........|  43
query             0 GNTAVAKCNMNHDEEFCDMLRLIDYNKAALTKFKTDVESALHL  43


Peptide ID: XDC94394.1|precursor|Mammarenavirus_sp.|EriAreV/RU/MOS/2022/hed22_15/S-v1|Erinaceus_sp.|Russia|2022|S|?|tile_5
Peptide: ASELKCFGNTAIAKCNTNHDEEFCDMLSLIEYNKKALIQFKDNLRE | Score: 160.0
target          287 AELKCFGNTAVAKCNEKHDEEFCDMLRLFDFNKQAIQRLK 327
                  0 .|||||||||.||||..|||||||||.|...||.|....|  40
query             1 SELKCFGNTAIAKCNTNHDEEFCDMLSLIEYNKKALIQFK  41


Peptide ID: XBA96065.1|precursor|Mammarenavirus_lassaense|LF2350|Homo_sapiens|Sierra_Leone|2012|S|?|tile_14
Peptide: KCNEKHDEEFCDMLRLFDFNKQAIQRLKAEAQMSIQLINKAVNALI | Score: 235.0
target          299 KCNEKHDEEFCD

{'pearsons_r': np.float64(0.9663135495366442),
 'top_peptides': ['GNTAVAKCNMNHDEEFCDMLRLIDYNKAALTKFKTDVESALHLFKT',
  'ASELKCFGNTAIAKCNTNHDEEFCDMLSLIEYNKKALIQFKDNLRE',
  'KCNEKHDEEFCDMLRLFDFNKQAIQRLKAEAQMSIQLINKAVNALI',
  'KCNQQHDEEFCDMLRLFDFNKEAINRLRVEAEKSINLINKAVNSLI',
  'VAKCNLNHDEEFCDMLRLIDFNKQALHTFKGEVKAGLQLITKAVNG',
  'AIAKCNLNHDEEFCDMLRLFDFNKQAIDKLRIEAQRGLNLITKAVN',
  'CNEKHDEEFCDMLRLFDFNKQAIRRLKTEAQMSIQLINKAVNALIN',
  'TAVAKCNLNHDEEFCDMLRLIDFNKQAVRRLKNETQLGLRLVTKAV',
  'KCNQQHDEEFCDMLRLFDFNREAISRLKVEAERSINLINKAVNSLI',
  'KCNEKHDEEFCDMLRLFDFNKQAIKRLKTEAQMSIQLINKAVNALI',
  'CNEKHDEEFCDMLRLFDFNKQAIKRLKTEAQMNIQLINKAVNALIN',
  'VAKCNLNHDEEFCDMLRLIDFNKQAVNKFKMEAQNSLNLITKAVNA',
  'VAKCNLNHDEEFCDMLRLFDFNKQAIDKLRIEAQRGLNLITKAVNS',
  'CFDNAAIAKCNKEHDEEFCDMLRLFDYNKASIAKLRGEASSSINLL',
  'TAIAKCNLNHDEEFCDMLRLIDFNKQALKTFKSEVNHGLQLITKAI',
  'IAKCNQNHDSEFCDMLRLFDYNKNAIKTLNEETKTRVNVLSHTINA',
  'GNTAVAKCNMNHDEEFCDMLRLIDYNKAALSKFKEDVESALHLFKV',
  'AELKCFGNTAVAKCNEKHDEEFCDMLRLFDFNKQAISRLRSEAQMS',
 

In [211]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='36-9F', literature_note = "Literature does not report the exact binding footprint of LASV 36.9F. 36.9F neutralized LASVpp but did not neutralize authentic LASV. ")

Correlation too low, Analysis stopped


{'pearsons_r': np.float64(-0.0025138759607037276),
 'passed_correlation': False,
 'top_peptides': None,
 'mapping': None,
 'covered_regions': [],
 'positions_covered': []}

In [212]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='37-2D_', literature_note = "Literature reports that LASV 37.2D targets the GPC-B quaternary epitope \
Key contacts are: GP2 fusion peptide, specifically contacts S269 and K272 \
the GP2 HR1 helix, directly interacts with K320 and Q324 (including hydrogen bonding and a salt bridge involving K320) \
Additional site: GP2 region around N395 also implicated to be important, in addition to broader GPC-B contacts spanning N-terminal loop, T-loop, fusion pep, HR1/HR2 regions")

Correlation too low, Analysis stopped


{'pearsons_r': np.float64(0.13408053843982484),
 'passed_correlation': False,
 'top_peptides': None,
 'mapping': None,
 'covered_regions': [],
 'positions_covered': []}

In [213]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='37-2G', literature_note = "Literature reports that LASV 37.2G targets the GPC-B quaternary epitope \
No atomic structure published, belongs to GPC-B competition groups, \n \
which has key amino acid positions aa62-63 (gp1 n-terminal loop), aa269-276 (fusion pep), aa324-325 (hr1) aa387-408 (t-loop + hr2),")


Peptide ID: XNW50201.1|precursor|Pannonia_mammarenavirus|PANV/ER15/2023/HUN|Erinaceus_roumanicus_(Northern_white-breasted_hedgehog)|Hungary|Jun-2023|S|?|tile_4
Peptide: ELKGSQFNLTMPTLCTKNDTHHYIKISNNSGLEVTLTNSSLVNKNF | Score: 147.0
target           71 ELNMETLNMTMPLSCTKNNSHHYIMVGNETGLELTLTNTSIINHKF 117
                  0 ||.....|.|||..||||..||||...|..|||.||||.|..|..|  46
query             0 ELKGSQFNLTMPTLCTKNDTHHYIKISNNSGLEVTLTNSSLVNKNF  46


Peptide ID: XBP46459.1|glycoprotein|Trematomus_arenavirus|TRD_L|Trematomus_lepidorhinus|Antarctica|Dec-2021/Jan-2022|?|?|tile_28
Peptide: MTFLPKPESNIIEPVVRFDSQHPAVQFDSQGLALPKPVPKQRTLIS | Score: 23.0
target          459 GKSCPKP 466
                  0 |...|||   7
query            30 GLALPKP  37


Peptide ID: WVH10120.1|precursor|Mammarenavirus_choriomeningitidis|KS20-3122_L5992|Apodemus_sylvaticus|Germany|2020|S|?|tile_17
Peptide: YCNYSKFWYLEHSKTNETSVPKCWLVSNGSYLNETHFNDQIEQEAD | Score: 220.0
target          362 YCNYSKYWYLNHTTTGRTSLPKCWLVSNGSYLNETHF

{'pearsons_r': np.float64(0.9747185393686572),
 'top_peptides': ['ELKGSQFNLTMPTLCTKNDTHHYIKISNNSGLEVTLTNSSLVNKNF',
  'MTFLPKPESNIIEPVVRFDSQHPAVQFDSQGLALPKPVPKQRTLIS',
  'YCNYSKFWYLEHSKTNETSVPKCWLVSNGSYLNETHFNDQIEQEAD',
  'YWYLNHTITGRTSLPKCWLVSNGSYLNETHFSDDIEQQADNMITEM',
  'KFWYLNNTKTGQVSLPQCWLISNGSYLNKTDFTDDIEREANNLVTE',
  'GSVQTGLTYQQLQTLHELLKPLDPSSETWMDIEGRHNDPVEICLFQ',
  'ACMAKQGQFELNDVVQSLTDLGLIYTAKYPNANDLEKLTKGHPVLG',
  'TVQYNLSHSTAVDAANHCGTLANGVLDAFYRFQWGRNISHVYNIPY',
  'AVAKCNMDHDEEFCDMLKLIDYNKAALTKFKKDVESALHLFKVTMN',
  'AIKRLRIEANKSTNLITSAVNSLSNYQLISKNHLRDLMGIPYCNYS',
  'GSFLNETEFSTDIEQEANNLITEMLQKEYIERQGKTPLGLVDIFIF',
  'FNGGNITVQYNLSHGTAHAAAKHCGSVANGVLYAFYKMFWSRNISD',
  'LGIPYCNYTKFWYLNSTRTGHHSLPKCWMISNGSYLNETHFSTEIE',
  'VEHDRQVLNNLSNCVHPAVKHRSRMVGLLEWIFRALKYDFNHDPTP',
  'RMVGLLEWIFRALKYDFNHDPTPLCQKETSTVNETRVQINITEGYW',
  'ALSSGSRIENNLEPGCDSNVGLFGHSTGTDYGWGLANFFSAGITNS',
  'IAKCNQNHDSEFCDMLRLFDYNKNAIKTLNEETKTRVNVLSHTINA',
  'TSLPKCWLVSNGSYLNETHFSDDIEQQADNMITEMLQKEYMDRQGK',
 

In [214]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='37-7H', literature_note = "Literature reports that LASV")

Correlation too low, Analysis stopped


{'pearsons_r': np.float64(0.5096684373099812),
 'passed_correlation': False,
 'top_peptides': None,
 'mapping': None,
 'covered_regions': [],
 'positions_covered': []}

### GPC-B group, non neutralizing

In [215]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='20-5G', literature_note = "Literature reports that LASV 20.5G is non-neutralizing \n \
atomic structures have not been published. It does target a conformational epitope, as it belongs to the GPC-B competition group. \n \
Robinson and colleagues. Binding is disrupted, when aa 362-366, aa400-412 and aa60-75 are deleted.")


Peptide ID: XOB76280.1|glycoprotein|Mammarenavirus_ryukyuense|CXMIX-YNSF117|Mus_musculus|China:_Yunnan|2021|S|?|tile_14
Peptide: GNTAVAKCNMNHDEEFCDMLRLIDYNKAALTKFKTDVESALHLFKT | Score: 151.0
target          293 GNTAVAKCNEKHDEEFCDMLRLFDFNKQAIQRLKAEAQMSIQL 336
                  0 |||||||||..|||||||||||.|.||.|....|........|  43
query             0 GNTAVAKCNMNHDEEFCDMLRLIDYNKAALTKFKTDVESALHL  43


Peptide ID: XNW50209.1|precursor|Pannonia_mammarenavirus|PANV/ER33/2023/HUN|Erinaceus_roumanicus_(Northern_white-breasted_hedgehog)|Hungary|Aug-2023|S|?|tile_13
Peptide: EKWMLVASELKCFGNTAVAKCNLNHDEEFCDMLSLIEYNKKALTQF | Score: 180.0
target          281 RWMLIEAELKCFGNTAVAKCNEKHDEEFCDMLRLFDFNKQAI 323
                  0 .|||...||||||||||||||..|||||||||.|...||.|.  42
query             1 KWMLVASELKCFGNTAVAKCNLNHDEEFCDMLSLIEYNKKAL  43


Peptide ID: XDC94394.1|precursor|Mammarenavirus_sp.|EriAreV/RU/MOS/2022/hed22_15/S-v1|Erinaceus_sp.|Russia|2022|S|?|tile_5
Peptide: ASELKCFGNTAIAKCNTNHDEEFCDMLSLIEYNKK

{'pearsons_r': np.float64(0.982270646229705),
 'top_peptides': ['GNTAVAKCNMNHDEEFCDMLRLIDYNKAALTKFKTDVESALHLFKT',
  'EKWMLVASELKCFGNTAVAKCNLNHDEEFCDMLSLIEYNKKALTQF',
  'ASELKCFGNTAIAKCNTNHDEEFCDMLSLIEYNKKALIQFKDNLRE',
  'KCNEKHDEEFCDMLRLFDFNKQAIQRLKAEAQMSIQLINKAVNALI',
  'KCNQQHDEEFCDMLRLFDFNKEAINRLRVEAEKSINLINKAVNSLI',
  'VAKCNLNHDEEFCDMLRLIDFNKQALHTFKGEVKAGLQLITKAVNG',
  'AVAKCNMDHDEEFCDMLKLIDYNKAALTKFKKDVESALHLFKVTMN',
  'AIAKCNLNHDEEFCDMLRLFDFNKQAIDKLRIEAQRGLNLITKAVN',
  'CNEKHDEEFCDMLRLFDFNKQAIRRLKTEAQMSIQLINKAVNALIN',
  'TAVAKCNLNHDEEFCDMLRLIDFNKQAVRRLKNETQLGLRLVTKAV',
  'KCNEKHDEEFCDMLRLFDFNKQAIKRLKTEAQMSIQLINKAVNALI',
  'CNEKHDEEFCDMLRLFDFNKQAIKRLKTEAQMNIQLINKAVNALIN',
  'VAKCNLNHDEEFCDMLRLIDFNKQAVNKFKMEAQNSLNLITKAVNA',
  'VAKCNLNHDEEFCDMLRLFDFNKQAIDKLRIEAQRGLNLITKAVNS',
  'CFDNAAIAKCNKEHDEEFCDMLRLFDYNKASIAKLRGEASSSINLL',
  'TAIAKCNLNHDEEFCDMLRLIDFNKQALKTFKSEVNHGLQLITKAI',
  'IAKCNQNHDSEFCDMLRLFDYNKNAIKTLNEETKTRVNVLSHTINA',
  'GNTAVAKCNMNHDEEFCDMLRLIDYNKAALSKFKEDVESALHLFKV',
  

In [216]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='38-8B', literature_note = "Literature reports that LASV 38.8B ")


Peptide ID: XNW50201.1|precursor|Pannonia_mammarenavirus|PANV/ER15/2023/HUN|Erinaceus_roumanicus_(Northern_white-breasted_hedgehog)|Hungary|Jun-2023|S|?|tile_4
Peptide: ELKGSQFNLTMPTLCTKNDTHHYIKISNNSGLEVTLTNSSLVNKNF | Score: 147.0
target           71 ELNMETLNMTMPLSCTKNNSHHYIMVGNETGLELTLTNTSIINHKF 117
                  0 ||.....|.|||..||||..||||...|..|||.||||.|..|..|  46
query             0 ELKGSQFNLTMPTLCTKNDTHHYIKISNNSGLEVTLTNSSLVNKNF  46


Peptide ID: XEM48943.1|precursor|Mammarenavirus_tacaribeense|DR011|bat|Dominican_Republic|04-Feb-2014|S|?|tile_9
Peptide: HYGWRLKNGLDHIYRGREEPCIEGERCLIKIQPEDWPVDCKADHTN | Score: 25.0
target          261 FTWTLSDS-----EGKDTP---GGYCLTR 282
                  0 ..|.|...-----.|...|---|..||..  29
query             1 YGWRLKNGLDHIYRGREEPCIEGERCLIK  30


Peptide ID: XBP46459.1|glycoprotein|Trematomus_arenavirus|TRD_L|Trematomus_lepidorhinus|Antarctica|Dec-2021/Jan-2022|?|?|tile_10
Peptide: VYGTLKPPTGTDDKLVRYHEIKRRSCLCDPKNLRTGPVEPPFEVFA | Score: 22.0
target 

{'pearsons_r': np.float64(0.9523977056792157),
 'top_peptides': ['ELKGSQFNLTMPTLCTKNDTHHYIKISNNSGLEVTLTNSSLVNKNF',
  'HYGWRLKNGLDHIYRGREEPCIEGERCLIKIQPEDWPVDCKADHTN',
  'VYGTLKPPTGTDDKLVRYHEIKRRSCLCDPKNLRTGPVEPPFEVFA',
  'QNRTWENHCEYASPIGFSRILLAQEKTKFLTRRLAGTFSWTLSDSS',
  'ALELNMQALNKTMPLSCSKNNSHHYIMVGNETGLELTLTNKSIIEH',
  'MGQIVTFFQEIPHILEEVMNIVLMTLSLIAILKGIYNFVTCGLGGL',
  'RTNISGRSWENTIVDLGGVPPNPGLNGSPSKQKGPVGVTAGLSYSQ',
  'CIALHNKNMLTHHSLALSSGSKVDNTLEPGCDSNVGLFGHSTGTDY',
  'HLSIPNFNQYEAMSCDFNNGKISIQYNLSHSTVTESANHCGTVANG',
  'LVRGPFQAHWGIDLAVGSTTVTIENATGTYALKTKNFTGCFKSNHD',
  'SFIGIVPNETFPLFDFSIRAPNDSAPVGRTLLITHDMKLFGHEAKN',
  'SPVTFSLHTPGCDSPEAFGMVGDKRVCLDPNGIQMNITKWMHGAMN',
  'GEIRDHVIEHEMALNYLLAHQGGLCSVVKGPMCCSDIDDFRQNVSD',
  'GLRLTKAEKAFNNTHLCVMRNESADLTHYLRIKHTHNNGEYYHGCE',
  'NDWLLESDHLISQILSREYEERQSKTPLGLVDLCFWCTLFYVSTIF',
  'LHLVKIPTHRHVKGGTCPKPHRLTNKGICSCGAFKVPGVKTVWKRR',
  'MGQIVTLVEALPHVIDEAINIAIIALIIITGVKAVYNFATCGIIAL',
  'SLNNSHHYINMGKSGLELTFTNNSILTHQQCNLTDAFKRNTFDHTI',
 

In [217]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='9-7A', literature_note = "Literature reports that LASV 9.7A ")

Correlation too low, Analysis stopped


{'pearsons_r': np.float64(0.00906379706011235),
 'passed_correlation': False,
 'top_peptides': None,
 'mapping': None,
 'covered_regions': [],
 'positions_covered': []}

### now GP1-A and GPC-A competition groups, both neutralizing

In [219]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='12-1F', literature_note = "Literature reports that LASV  ")

Correlation too low, Analysis stopped


{'pearsons_r': np.float64(0.06345166055986518),
 'passed_correlation': False,
 'top_peptides': None,
 'mapping': None,
 'covered_regions': [],
 'positions_covered': []}

In [220]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='25-10C_', literature_note = "Literature reports that LASV  ")

Correlation too low, Analysis stopped


{'pearsons_r': np.float64(0.01222588474992132),
 'passed_correlation': False,
 'top_peptides': None,
 'mapping': None,
 'covered_regions': [],
 'positions_covered': []}

In [ ]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='36-1F', literature_note = "Literature reports that LASV LASV36.1F \n \
binds a quarternary epitope bridging GP1 and GP2 subunits, but predominantly GP1. \n \
Key residues are aa95 and aa228. putative epitope regions are aa62-68, gp2 regions aa270-278 \n \
there is a structure available - check. \n \
INterestingly, it does NOT bind the regions where the peptides enriched here map to!!! ")


Peptide ID: XOB76280.1|glycoprotein|Mammarenavirus_ryukyuense|CXMIX-YNSF117|Mus_musculus|China:_Yunnan|2021|S|?|tile_14
Peptide: GNTAVAKCNMNHDEEFCDMLRLIDYNKAALTKFKTDVESALHLFKT | Score: 151.0
target          293 GNTAVAKCNEKHDEEFCDMLRLFDFNKQAIQRLKAEAQMSIQL 336
                  0 |||||||||..|||||||||||.|.||.|....|........|  43
query             0 GNTAVAKCNMNHDEEFCDMLRLIDYNKAALTKFKTDVESALHL  43


Peptide ID: XDC94394.1|precursor|Mammarenavirus_sp.|EriAreV/RU/MOS/2022/hed22_15/S-v1|Erinaceus_sp.|Russia|2022|S|?|tile_5
Peptide: ASELKCFGNTAIAKCNTNHDEEFCDMLSLIEYNKKALIQFKDNLRE | Score: 160.0
target          287 AELKCFGNTAVAKCNEKHDEEFCDMLRLFDFNKQAIQRLK 327
                  0 .|||||||||.||||..|||||||||.|...||.|....|  40
query             1 SELKCFGNTAIAKCNTNHDEEFCDMLSLIEYNKKALIQFK  41


Peptide ID: XBA96065.1|precursor|Mammarenavirus_lassaense|LF2350|Homo_sapiens|Sierra_Leone|2012|S|?|tile_14
Peptide: KCNEKHDEEFCDMLRLFDFNKQAIQRLKAEAQMSIQLINKAVNALI | Score: 235.0
target          299 KCNEKHDEEFCD

{'pearsons_r': np.float64(0.9556554964288168),
 'top_peptides': ['GNTAVAKCNMNHDEEFCDMLRLIDYNKAALTKFKTDVESALHLFKT',
  'ASELKCFGNTAIAKCNTNHDEEFCDMLSLIEYNKKALIQFKDNLRE',
  'KCNEKHDEEFCDMLRLFDFNKQAIQRLKAEAQMSIQLINKAVNALI',
  'KCNQQHDEEFCDMLRLFDFNKEAINRLRVEAEKSINLINKAVNSLI',
  'VAKCNLNHDEEFCDMLRLIDFNKQALHTFKGEVKAGLQLITKAVNG',
  'AVAKCNMDHDEEFCDMLKLIDYNKAALTKFKKDVESALHLFKVTMN',
  'AIAKCNLNHDEEFCDMLRLFDFNKQAIDKLRIEAQRGLNLITKAVN',
  'CNEKHDEEFCDMLRLFDFNKQAIRRLKTEAQMSIQLINKAVNALIN',
  'TAVAKCNLNHDEEFCDMLRLIDFNKQAVRRLKNETQLGLRLVTKAV',
  'KCNEKHDEEFCDMLRLFDFNKQAIKRLKTEAQMSIQLINKAVNALI',
  'CNEKHDEEFCDMLRLFDFNKQAIKRLKTEAQMNIQLINKAVNALIN',
  'AKCNEAHDEEFCDMLRLFDYNKEAIKRFKGETQAALHHFKEAINAL',
  'VAKCNLNHDEEFCDMLRLIDFNKQAVNKFKMEAQNSLNLITKAVNA',
  'VAKCNLNHDEEFCDMLRLFDFNKQAIDKLRIEAQRGLNLITKAVNS',
  'CFDNAAIAKCNKEHDEEFCDMLRLFDYNKASIAKLRGEASSSINLL',
  'TAIAKCNLNHDEEFCDMLRLIDFNKQALKTFKSEVNHGLQLITKAI',
  'IAKCNQNHDSEFCDMLRLFDYNKNAIKTLNEETKTRVNVLSHTINA',
  'GNTAVAKCNMNHDEEFCDMLRLIDYNKAALSKFKEDVESALHLFKV',
 

In [ ]:
mAb13_4e_avg = mAb13_4e.mean(axis=1)

mAb13_4e = mAb13_4e_avg.sort_values(ascending=False).head(20)

mAb13_4e_peptide_sequences = peps_metadata.loc[peps_metadata['seq_id'].isin(mAb13_4e.index.to_list())]

In [ ]:
mAb13_4e = mAb13_4e.sort_values(ascending=False).head(20)
mAb13_4e_peptide_sequences = peps_metadata.loc[peps_metadata['seq_id'].isin(mAb13_4e.index.to_list())]

#### Aligning to Josiah LASV GPC refseq

In [ ]:
from Bio.Align import PairwiseAligner, substitution_matrices
from Bio import Align, SeqIO

ref_path = "../ref_seqs/UP000002473_11622_LASV_Josiah_reference.fasta"

#SeqIO.to_dict() expects parsed sequence records
ref_records = SeqIO.to_dict(SeqIO.parse(ref_path, 'fasta'))

target_id = "sp|P08669|GLYC_LASSJ"
ref_seq = ref_records[target_id].seq

#configure pairwise aliger for peptides 
aligner = Align.PairwiseAligner()
aligner.mode = 'local' #for peptides
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

#align each peptide to the reference sequence
peptides = mAb13_4e_peptide_sequences['sequence'].to_list()

positions_covered = []

for pep in peptides:
    alignments = aligner.align(ref_seq, pep)
    best_alignment = alignments[0] #top scoring alignment

    print(f"Peptide: {pep} | Score: {best_alignment.score}")

    #print start and end coordinated on the reference protein (0-based)
    ref_start, ref_end = best_alignment.aligned[0][0]
    print(f"Mapped to reference positions {ref_start + 1} to {ref_end}")
    print(best_alignment)

    for i in range(ref_start+1, ref_end):
        if i in positions_covered:
            continue
        else:
            positions_covered.append(i)

positions_covered = sorted(positions_covered)

Peptide: KFWYLNNTKTGQVSLPQCWLISNGSYLNETHFSADIEREANNLITE | Score: 204.0
Mapped to reference positions 368 to 413
target          367 KYWYLNHTTTGRTSLPKCWLVSNGSYLNETHFSDDIEQQADNMITE 413
                  0 |.||||.|.||..|||.|||.||||||||||||.|||..|.|.|||  46
query             0 KFWYLNNTKTGQVSLPQCWLISNGSYLNETHFSADIEREANNLITE  46

Peptide: YCNYSKFWYLEHAKTGETSVPKCWLVTNGSYLNETHFSDQIEQEAD | Score: 225.0
Mapped to reference positions 363 to 408
target          362 YCNYSKYWYLNHTTTGRTSLPKCWLVSNGSYLNETHFSDDIEQQAD 408
                  0 ||||||.|||.|..||.||.||||||.||||||||||||.|||.||  46
query             0 YCNYSKFWYLEHAKTGETSVPKCWLVTNGSYLNETHFSDQIEQEAD  46

Peptide: KELLGIPYCNYTKFWYLNNTSLNLHSLPKCWLVSNNSYLNKTEFSF | Score: 188.0
Mapped to reference positions 356 to 400
target          355 RDIMGIPYCNYSKYWYLNHTTTGRTSLPKCWLVSNGSYLNETHFS 400
                  0 ....|||||||.|.||||.|.....||||||||||.||||.|.||  45
query             0 KELLGIPYCNYTKFWYLNNTSLNLHSLPKCWLVSNNSYLNKTEFS  45

Peptide: NDQLIMKNHLRDIMGI